> 🚨 **Folders required before running this Notebook: Building Block `.xyz` Files**

- `NODE_FILES` *(.xyz, required)*  
  Files containing the node fragments / higher-connectivity building units.  
  **Location:** `0_node/` directory  
  **Default behavior:** The required number of node files depends on the selected `TOPO`:  
  - `hcb` / `sql` / `kgm`: exactly one `.xyz` file must be present  
  - `hcb_ab`: exactly two `.xyz` files must be present  
  Otherwise, provide explicit paths via `input_nodes=[...]`.

- `LINKER_FILES` *(.xyz, topology-dependent)*  
  Files containing the linker fragments / lower-connectivity building units.  
  **Location:** `0_linker/` directory  
  **Default behavior:** The required number of linker files depends on the selected `TOPO`:  
  - `hcb` / `sql` / `kgm`: exactly one `.xyz` file must be present  
  - `hcb_ab`: no linker file is required; files in `0_linker/` are ignored  
  Pass `input_linkers=[]` to explicitly provide no linkers, or provide explicit paths via `input_linkers=[...]`.

> 🚨 **Building Block Requirements**
>
> - **Dummy atoms are required at every intended connection site.**  
>   Each position at which a bond will be formed in the final COF must be marked by a dummy atom in the input building block.
>
> - **Dummy atom encoding**
>   - `He` = connection site (all bonds are treated as single)
>
> - **Important**  
>   Dummy atoms are used only to define the connection sites during construction. They do not represent real atoms in the final COF structure.


In [11]:
import coflandscaper as cl

### General Settings & Options

#### Structural Parameters

- `TOPOLOGY` *(str, required, default: None)*  
  Defines the underlying network topology.  
  **Allowed values:** `hcb`, `sql`, `hcb_ab`, `kgm`  
  - `hcb` / `sql` / `kgm`: require one node file and one linker file  
  - `hcb_ab`: requires two node files and no linker file

#### Naming Parameters

- `COF_NAME` *(str, required, default: None)*  
  Unique identifier for the generated structure.  
  All output files and directories will be named using this value.  
  **Constraints:**  
  - Should be unique within the working directory.  
  - Avoid spaces and special characters for compatibility.

#### Stacking Parameters

- `MODE` *(str, required, default: None)*  
  Defines the stacking mode(s) to be generated.  
  **Allowed values:**  
  - `incl` → inclined stacking  
  - `serr` → serrated stacking  
  - `both` → generate both stacking modes

  **Behavior:**  
  Selecting `both` will generate both configurations sequentially and store them in the corresponding output structure.


In [12]:
TOPOLOGY = "hcb"
COF_NAME = "COF-1"
MODE = "both"

#### Optional: MACE Head Configuration

- `MACE_HEAD` *(str, optional, default: `spice_wB97M` )*  
  Specifies the prediction head used by the `mace-mh-1` model.  

  **Allowed values (with Dispersion Information):**  
  - `omat_pbe` -> D3 on, `dispersion_xc="pbe"`, `dispersion_cutoff=21.167088422553647`  
  - `matpes_r2scan` -> D3 on, `dispersion_xc="r2scan"`, `dispersion_cutoff=40.0`  
  - `omol` -> D3 off  
  - `spice_wB97M` -> D3 off

  **Reference:**  
  Model details: https://huggingface.co/mace-foundations/mace-mh-1  

  **Usage:**  
  Set `MACE_HEAD` in the subsequent cell to override the default value.

In [13]:
MACE_HEAD = "spice_wB97M"

### Construction Parameters (Optional)

These parameters allow you to override the automatic file detection and manually specify the `.xyz` building block files to be used in COF construction.

#### File Overrides

- `input_nodes` *(list[str], optional)*  
  Explicit paths to the node (high-connectivity) `.xyz` files.

- `input_linkers` *(list[str], optional)*  
  Explicit paths to the linker (low-connectivity) `.xyz` files.  
  Use `[]` to specify that no linker files should be used (for `hcb_ab`).

If these are not provided, the notebook will search for `.xyz` files in the `0_node/` and `0_linker/` subfolders based on the selected topology.


Default

In [ ]:
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, cof_name=COF_NAME)

In [ ]:
preopt = cl.MaceOpt()
preopt.run_preopt(cof_name=COF_NAME)

Configurable

In [15]:
NODES = ["0_node/example_node.xyz"]
LINKERS = ["0_linker/example_linker.xyz"]

In [ ]:
builder = cl.BuildCOF2D()

builder.build(
    topo=TOPOLOGY,
    cof_name=COF_NAME,
    input_nodes=NODES,
    input_linkers=LINKERS,
)

In [ ]:
preopt = cl.MaceOpt(
    fmax=0.01,
    dtype="float64",
    head=MACE_HEAD,
    device="cpu",
)
preopt.run_preopt(
    cof_name=COF_NAME,
    fix_z=True,
    input_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_unopt.cif",
    output_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif",
)

### ILD × ILS Structure Matrix Generation

This step generates stacked COF structures by systematically varying:

- **Interlayer Distance (ILD)** (out-of-plane spacing)  
- **Interlayer Slipping (ILS)** (in-plane shift)  

The input structure is the pre-optimized single-layer (AA stacking with large interlayer distance).  
This step converts it into physically meaningful bulk stacking configurations.

---

#### Parameters

- `ild_start` *(float, optional, default: `3.0`)*  
  Start of ILD range (in A).

- `ild_end` *(float, optional, default: `4.0`)*  
  End of ILD range (in A).

- `ild_step` *(float, optional, default: `0.1`)*  
  Step size for ILD (in A).

- `ils_length_start` *(float, optional, default: `0.0`, equals AA-stacking)*  
  Start of ILS range (in A).

- `ils_length_end` *(float or None, optional, default: `None`)*  
  End of ILS range (in A). If `None`, defaults to the topology-derived AB shift length.

- `ils_length_step` *(float, optional, default: `1.0`)*  
  Step size for ILS (in A).

- `ils_angle` *(float or None, optional, default: `None`)*  
  Direction of ILS in degrees. If `None`, defaults to the topology-derived AB shift direction.

- `print_shift` *(bool, optional, default: `False`)*  
  If enabled, prints the auto-computed default shift length and angle.

  **Note:**   
  Manual modification is generally discouraged unless exploring non-standard stacking pathways.

---

#### Input / Output
- `cof_name` *(str, required)*  
  This parameter has no default; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This parameter has no default; pass `MODE` explicitly.

- `input_cif` *(str, optional, default: `{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif`)*  
  Path to the input structure file.

- `output_base_folder` *(str, optional, default: `{COF_NAME}/2_{COF_NAME}_matrix/`)*  
  Base directory for generated structures.  
  Generated stacked structures organized by mode:  
  - `.../serr/` → serrated stacking configurations  
  - `.../incl/` → inclined stacking configurations  

These structures are intended for subsequent single-point energy evaluations and energy landscape analysis.


Defaults

In [ ]:
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

Configurable

In [ ]:
matrix = cl.CreateMatrix(
    ild_start=3.0,
    ild_end=4.0,
    ild_step=0.1,
    ils_length_start=0.0,
    ils_length_end=None,
    ils_length_step=1.0,
    ils_angle=None,
    print_shift=True,
)
matrix.run(
    cof_name=COF_NAME,
    topo=TOPOLOGY,
    mode=MODE,
    input_cif=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif",
    output_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### MACE Single-Point Energy Evaluation

This step computes single-point energies for generated structures using `MaceSP` (no geometry optimization).

> 🚨 **Optional: DFT Pre-Screening Strategy**
>
> Use this stage as a pre-screening step for DFT: sample the full ILS/ILD space at low cost with MACE, then select a smaller set for higher-accuracy DFT.

---

#### Parameters

- `dtype` *(str, optional, default: `float64`)*  
  Numerical precision used during evaluation.  
  **Allowed values:** `float32`, `float64`

- `head` *(str, optional, default: `MACE_HEAD`)*  
  MACE head selection with fixed built-in settings.

- `device` *(str, optional, default: `cpu`)*  
  Compute device used for evaluation.  
  **Allowed values:** `cpu`, `cuda` *(if available)*

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_folder` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_csv_dir` *(str, optional, default: None)*  
  CSV files containing single-point energies default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`  

  **File naming:**  
  `{COF_NAME}_sp_energies_{serr|incl}.csv`

---

Default

In [ ]:
sp = cl.MaceSP()
sp.run_mode(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
sp = cl.MaceSP(
    device="cpu",
    dtype="float64",
    head=MACE_HEAD,
)
sp.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=None,
    output_csv_dir=None,
)

### DFT Single-Point Input Generation (CRYSTAL23)

This step generates CRYSTAL23 `.d12` input files for all stacked structures.

The generated inputs must be executed externally (on an HPC system).  
After completion, the corresponding `.out` files should be placed back into the same directories as their respective `.d12` files.

**Default level of theory:** `HSEsol-3c/sol-mSVP`

---

#### Parameters

- `basisset` *(str, optional, default: `SOLDEF2MSVP`)*  
  Basis set used for all atoms in the system.

- `functional` *(str, optional, default: `HSESOL3C`)*  
  Exchange–correlation functional.

- `shrink` *(str, optional, default: `2 2 8`)*  
  Defines the Monkhorst–Pack $k$-point sampling grid (`n1 n2 n3`), where larger values increase Brillouin zone sampling accuracy at higher computational cost.

- `post_block` *(str or None, optional, default: `None`)*  
  Optional override for the full CRYSTAL23 input tail.  
  If `None`, a standard `BASISSET` / `DFT` / `SHRINK` block is automatically generated.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.
  
- `input_base_folder` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_base_folder` *(str, optional, default: None)*  
  CRYSTAL23 `.d12` input files default written to:  
  `COF_NAME/2_{COF_NAME}_matrix/dft_{serr|incl}/`

Defaults

In [ ]:
sp = cl.CrystalSP()
sp.generate_input(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
sp = cl.CrystalSP(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    post_block=None,
)
sp.generate_input(
    cof_name=COF_NAME,
    mode=MODE,
    input_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
    output_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### CRYSTAL Energy Extraction

This step parses CRYSTAL23 `.out` files and extracts total energies for all generated structures.

Run this after all CRYSTAL jobs have completed and the `.out` files are placed alongside their corresponding `.d12` inputs.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_base_folder` *(str, optional)*  
  CRYSTAL23 output files (`.out`) default located in:  
  `COF_NAME/2_{COF_NAME}_matrix/dft_{serr|incl}/`

- `output_base_folder` *(str, optional)*  
  CSV file(s) containing extracted energies default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`

  The data is structured for subsequent energy landscape analysis.

Defaults

In [ ]:
sp = cl.CrystalSP()
sp.read_output(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
sp = cl.CrystalSP()
sp.read_output(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    input_base_folder=f"{COF_NAME}/2_{COF_NAME}_matrix/",
)

### Potential Energy Landscape (PES)

This step plots an approximate stacking potential energy surface by mapping the relative energies of the generated ILD/ILS structure matrix as a function of interlayer distance (ILD) and interlayer slipping (ILS).

Because no full structural relaxation is performed at this stage, the PES is treated as a reduced-dimensional screening model rather than a full description of the underlying high-dimensional energy landscape. Within this approximation, it provides a qualitative to semi-quantitative representation of the relative stability of different stacking arrangements and serves to identify candidate minima for subsequent refinement by full geometry optimization.

---

#### Parameters

- `colorscheme` *(str, optional, default: `viridis`)*  
  Matplotlib colormap used for visualization.  
  See: https://matplotlib.org/stable/users/explain/colors/colormaps.html

- `rel_energy_max` *(float or None, optional, default: `None`)*  
  Upper cutoff for relative energies (in eV); values above this threshold are clipped in the plots.

- `show_minima_markers` *(bool, optional, default: `True`)*  
  If enabled, highlights minima on the PES (global minimum in red; local minima in orange when `minima_mode="local"`).

- `minima_mode` *(str, optional, default: `global`)*  
  Controls minima handling:  
  - `global` -> mark only one global minimum  
  - `local` -> mark all local minima  

- `show_title_block` *(bool, optional, default: `False`)*  
  If enabled, draws the title and two header lines (stacking mode and level of theory).

- `show` *(bool, optional, default: `False`)*  
  If `True`, displays interactive plot windows.  
  Keep `False` for HPC runs to avoid blocking or backend issues.

- `dft` *(bool, optional, default: `False`)*  
  If enabled, reads `_dft` CSVs and labels the plot title/level of theory as `DFT`.  
  Otherwise, reads `MACE-MH-1` CSVs.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_folder` *(str, optional, default: None)*  
  Energy data in `{COF_NAME}_sp_energies_{serr|incl}.csv` or default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/{COF_NAME}_sp_energies_{serr|incl}`  

- `output_folder` *(str, optional, default: None)*  
  PES plots `pes_{COF_NAME}_{serr|incl}.png` default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`

Defaults

In [ ]:
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE, show=True)

Configurable

In [ ]:
landscape = cl.Landscape()
landscape.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    output_folder=f"{COF_NAME}/3_{COF_NAME}_landscape",
    colorscheme="viridis",
    rel_energy_max=None,
    show_minima_markers=True,
    minima_mode="global",
    show_title_block=False,
    show=True,
    dft=True,
)

### Structure Selection for Optimization

This step selects candidate structures corresponding to automatically detected local or gloabl minima are copies them into dedicated folders.

---

#### Additional PES Sampling Points

This allows manual inclusion of additional (ILD, ILS) points from the ILS × ILD matrix in the selection process.
The specified strcutures can appended to or used instead of the automatically detected minima for each stacking mode.

---

- `EXTRA_SERR` *(list[tuple[float, float]], optional)*  
  Additional points for serrated stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

- `EXTRA_INCL` *(list[tuple[float, float]], optional)*  
  Additional points for inclined stacking.  
  Format: `[(ILD, ILS), ...]` in Å  

---

#### Parameters

- `selections_serr` *(list[tuple[float, float]], optional, default: `None`)*  
  Additional (ILD, ILS) points for serrated stacking to include in the selection.

- `selections_incl` *(list[tuple[float, float]], optional, default: `None`)*  
  Additional (ILD, ILS) points for inclined stacking to include in the selection.

- `include_autoselect` *(bool, optional, default: `True`)*  
  Includes automatically detected minima from the PES.

- `autoselect_minima` *(str, optional, default: `global`)*  
  Which minima type is used by auto-selection.  
  **Allowed values:** `global`, `local`

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_base_folder` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/2_{COF_NAME}_matrix/{serr|incl}/`

- `output_base_folder` *(str, optional, default: None)*  
  Selected structures (`.cif`) default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

Defaults, auto-selected global minima structures

In [ ]:
selector = cl.SelectCofs()
selector.run_mode(cof_name=COF_NAME, mode=MODE)

Additional Candidate Structures  
This example uses an additional candidate structure identical to the automatically selected global minimum and is included only to illustrate the corresponding workflow syntax.

In [ ]:
EXTRA_SERR = [(3.3, 2.0)]
EXTRA_INCL = [(3.3, 2.0)]

Configurable

In [ ]:
selector = cl.SelectCofs()
selector.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    selections_serr=EXTRA_SERR,
    selections_incl=EXTRA_INCL,
    include_autoselect=True,
    autoselect_minima="global",
    input_base=f"{COF_NAME}/2_{COF_NAME}_matrix",
    output_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
)

### MACE Geometry Optimization

This step performs geometry optimizations of selected stacking structures using `MaceOpt`.

`MaceOpt` uses ASE `FrechetCellFilter` + `LBFGS`, writes optimized CIF files, and can write a combined energy CSV (`{COF_NAME}_opt_energies_per_layer.csv`).

---

#### Parameters

- `fmax` *(float, optional, default: `0.01`)*  
  Force convergence threshold for geometry optimization in `eV/Å`.

- `max_steps` *(int, optional, default: `2000`)*  
  Maximum number of optimizer iterations per structure.

- `dtype` *(str, optional, default: `float64`)*  
  Numerical precision used during optimization.  
  **Allowed values:** `float32`, `float64`

- `head` *(str, optional, default: `MACE_HEAD`)*  
  MACE prediction head used for energy and force evaluation.  

- `device` *(str, optional, default: `cpu`)*  
  Compute device used for the calculation.  
  **Allowed values:** `cpu`, `cuda` *(if available)*

- `fix_z` *(bool, optional, default: `False` for the optimization workflow)*  
  Constrain atomic motion along the $z$-axis during relaxation when set to `True`.  
  Use `fix_z=True` only for the preoptimization stage.

- `save_opt_energies_csv` *(bool, optional, default: `True`)*  
  If `True`, writes `COF_NAME/4_{COF_NAME}_optimization/{COF_NAME}_opt_energies_per_layer.csv`.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_base` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

- `output_base` *(str, optional, default: None)*  
  Optimized structures (`.cif`) and optional energies CSV are written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

Defaults

In [ ]:
opt = cl.MaceOpt()
opt.run_mode(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
opt = cl.MaceOpt(
    fmax=0.01,
    dtype="float64",
    head=MACE_HEAD,
    device="cpu",
    fix_z=False,
)
opt.run_mode(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
    output_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    save_opt_energies_csv=True,
)

### DFT Geometry Optimization Input Generation (CRYSTAL23)

This step generates CRYSTAL23 `.d12` input files for geometry optimization of the selected structures.

The generated inputs must be executed externally (on an HPC system).  
After completion, the corresponding `.out` files should be placed back into the same directories as their respective `.d12` files.

---

#### Parameters

- `basisset` *(str, optional, default: `SOLDEF2MSVP`)*  
  Basis set used for all atoms in the system.

- `functional` *(str, optional, default: `HSESOL3C`)*  
  Exchange–correlation functional.

- `shrink` *(str, optional, default: `2 2 8`)*  
  Defines the Monkhorst–Pack $k$-point sampling grid (`n1 n2 n3`), where larger values increase Brillouin zone sampling accuracy at higher computational cost.

- `maxtradius` *(str, optional, default: `0.5` as per default in Crystal23)*  
  The trust radius limits the step length of the displacement at each cycle, according to the quadratic form of the surface in the actual region.

- `post_block` *(str or None, optional, default: `None`)*  
  Optional override for the full CRYSTAL23 input tail.  
  If `None`, a standard input block is automatically generated.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_base` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  `COF_NAME/3_{COF_NAME}_landscape/selection/{serr|incl}/`

- `output_base` *(str, optional, default: None)*  
  CRYSTAL23 `.d12` files default written to:  
  `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/`

Defaults

In [ ]:
opt = cl.CrystalOpt()
opt.generate_input(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
opt = cl.CrystalOpt(
    basisset="SOLDEF2MSVP",
    functional="HSESOL3C",
    shrink="2 2 8",
    maxtradius="0.5",
    post_block=None,
)

opt.generate_input(
    cof_name=COF_NAME,
    mode=MODE,
    input_base_folder=f"{COF_NAME}/3_{COF_NAME}_landscape/selection",
    output_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
)

### Extract CRYSTAL23 Optimized Structures

This step extracts optimized structures from CRYSTAL23 `.out` files, converts them to `.cif` format,
and reads total energies to generate CSV files with relative energies per layer for comparison across stacking configurations.

Run this after all CRYSTAL geometry optimization jobs have completed and the `.out` files are placed alongside their corresponding `.d12` inputs.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_base_folder` *(str, optional, default: None)*  
  CRYSTAL23 output files (`.out`) default read from:  
  `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/`

- `output_base_folder` *(str, optional, default: None)*  
  Extracted structures (`.cif`) and CSV files with relative energies per layer default written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

Defaults

In [ ]:
opt = cl.CrystalOpt()
opt.extract_cif(cof_name=COF_NAME, mode=MODE)
opt.read_output(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
opt = cl.CrystalOpt()
opt.extract_cif(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    input_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
)
opt.read_output(
    cof_name=COF_NAME,
    mode=MODE,
    output_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    input_base_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
)

### Analysis & Visualization

This step analyzes optimized structures by computing interlayer distance (ILD) and interlayer slipping (ILS) from the optimized structures, and writing a summary CSV for comparison across stacking configurations.

In addition, structures can be visualized using an interactive viewer with configurable supercell sizes.

---

#### Parameters

- `dft` *(bool, optional, default: `False`)*  
  If enabled, uses DFT-optimized structures instead of MACE-optimized ones.

- `print_values` *(bool, optional, default: `True`)*  
  If enabled, prints computed ILD/ILS values to the console.

- `add_unit_cell` *(bool, optional, default: `True`)*  
  If enabled, displays the unit cell in the visualization.

- `supercell_size_serr` *(tuple[int, int, int], optional, default: `(2, 2, 1)`)*  
  Supercell size for serrated structures in the viewer.

- `supercell_size_incl` *(tuple[int, int, int], optional, default: `(2, 2, 2)`)*  
  Supercell size for inclined structures in the viewer.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.
  
- `input_base` *(str, optional, default: None)*  
  Structures (`.cif`) default read from:  
  - `COF_NAME/4_{COF_NAME}_optimization/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/` *(dft=True)*

- `output_base` *(str, optional, default: None)*  
  Analysis results (CSV) default written to `final_structures.csv` *(dft=False)* `dft_final_structures.csv` *(dft=True)*  or in:  
  `COF_NAME/5_{COF_NAME}_analysis/`  

Defaults

In [ ]:
analyzer = cl.AnalyzeStacking()
analyzer.analyze(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
analyzer = cl.AnalyzeStacking()
analyzer.analyze(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    output_base=f"{COF_NAME}/5_{COF_NAME}_analysis",
    dft=True,
    print_values=True,
)

Defaults

In [ ]:
visualizer = cl.VisualizeCOF()
visualizer.visualize_cof(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
visualizer = cl.VisualizeCOF()
visualizer.visualize_cof(
    cof_name=COF_NAME,
    mode=MODE,
    input_base=f"{COF_NAME}/4_{COF_NAME}_optimization",
    dft=True,
    add_unit_cell=True,
    supercell_size_serr=(2, 2, 1),
    supercell_size_incl=(2, 2, 2),
)

### PXRD Pattern Generation

This step simulates PXRD patterns from optimized CIF structures and writes per-structure `.xy` files.

---

#### Parameters

- `wavelength` *(str, optional, default: `CuKa`)*  
  X-ray source line used for simulation.  
  **Allowed values:** `CuKa`, `MoKa`, `CrKa`, `FeKa`, `CoKa`, `AgKa`  
  Choose this to match your instrument/source.

- `two_theta_range` *(tuple[float, float], optional, default: `(1.5, 60.0)`)*  
  Angular simulation window in degrees for generated `.xy` data.

- `dft` *(bool, optional, default: `False`)*  
  If enabled, uses DFT-optimized structures instead of MACE-optimized ones.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `input_folder` *(str, optional, default: None)*  
  Structure files (`.cif`) default read from:  
  - `COF_NAME/4_{COF_NAME}_optimization/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/4_{COF_NAME}_optimization/dft_{serr|incl}/` *(dft=True)*

- `output_folder` *(str, optional, default: None)*  
  PXRD data (`.xy`) default written to:  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy_dft/{serr|incl}/` *(dft=True)*

Default

In [ ]:
pxrd = cl.PXRD(wavelength="CuKa", two_theta_range=(1.5, 60.0))
pxrd.run(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
pxrd = cl.PXRD(
    wavelength="CuKa",
    two_theta_range=(1.5, 60.0),
)

pxrd.run(
    cof_name=COF_NAME,
    mode=MODE,
    dft=True,
    input_folder=f"{COF_NAME}/4_{COF_NAME}_optimization",
    output_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_xy_dft",
)

### PXRD Peak Extraction

This step extracts the strongest simulated PXRD peaks from generated `.xy` files and writes peak tables.

---

#### Parameters

- `max_peaks` *(int, optional, default: `10`)*  
  Maximum number of peaks reported per structure.

- `min_relative_intensity` *(float, optional, default: `1.0`)*  
  Minimum relative intensity threshold in %. Peaks below this value are ignored.

- `dft` *(bool, optional, default: `False`)*  
  If enabled, reads PXRD `.xy` files generated from DFT-optimized structures.

- `print_peaks` *(bool, optional, default: `True`)*  
  If enabled, prints the extracted peaks in the notebook or log output.

- `save_csv` *(bool, optional, default: `True`)*  
  If enabled, writes the extracted peaks to `pxrd_peaks.csv`.

---

#### Input / Output

- `cof_name` *(str, required)*  
  This parameter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This parameter has no default mode; pass `MODE` explicitly.

- `xy_folder` *(str, optional, default: None)*  
  PXRD `.xy` files default read from:  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_xy_dft/{serr|incl}/` *(dft=True)*

- `output_folder` *(str, optional, default: None)*  
  Peak tables default written to:  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_peaks/{serr|incl}/` *(dft=False)*  
  - `COF_NAME/5_{COF_NAME}_analysis/pxrd_peaks_dft/{serr|incl}/` *(dft=True)*

---

#### Output Table

The returned peak table contains one row per extracted peak:

- `structure`  
  Structure name derived from the `.xy` filename.

- `rank`  
  Peak rank after sorting by relative intensity.

- `two_theta_deg`  
  Peak position in degrees 2θ.

- `relative_intensity`  
  Relative peak intensity in %, normalized so that the strongest peak of each structure is 100%.

Default

In [ ]:
pxrd.extract_peaks(cof_name=COF_NAME, mode=MODE)

Configurable

In [ ]:
pxrd.extract_peaks(
    cof_name=COF_NAME,
    mode=MODE,
    dft=True,
    xy_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_xy_dft/",
    output_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_peaks_dft/",
    max_peaks=10,
    min_relative_intensity=1.0,
    print_peaks=True,
    save_csv=True,
)

### PXRD Plot

This step provides two plotting helpers:

- `plot_sim`  
  Stacked simulated PXRD patterns from `pxrd_xy/{serr|incl}` (or `pxrd_xy_dft/{serr|incl}` when `dft=True`).

- `plot_sim_vs_exp`  
  One experimental `.xy` file compared against every simulated `.xy` file in its own row.

#### Parameters

- `exp_xy_file` *(str or Path, optional, default: None)*  
  Path to the experimental `.xy` file.  
  **Default behavior:** If not provided, the folder `experimental_pxrd` (in the current directory) is searched for exactly one `.xy` file. If multiple `.xy` files exist, you must specify the exact path explicitly.  

- `xy_folder` for `plot_sim` or `simulated_xy_folder` for `plot_sim_vs_exp` *(str or Path, optional, default: None)*  
  Path to the simulated `.xy` files in their `MODE` subfolders.  
  **Custom labels:** To change the label displayed in the plot, simply rename your `.xy` file before passing it.

- `show` *(bool, optional, default: `True`)*  
  If enabled, displays the generated figure in the notebook/session.

- `save` *(bool, optional, default: `True`)*  
  If enabled, writes the `.png` figure to disk.

- `xlim` *(tuple[float, float], optional, default: `(1.5, 60.0)`)*  
  X-axis bounds as minimum and maximum 2θ range [º].

#### Output
- `cof_name` *(str, required)*  
  This paramter has no default mode; pass `COF_NAME` explicitly.

- `mode` *(str, required)*  
  This paramter has no default mode; pass `MODE` explicitly.

- `plot_sim` writes `{COF_NAME}_sim_{serr|incl}.png`.  
- `plot_sim_vs_exp` writes `{COF_NAME}_{serr|incl|both}.png`.

In [ ]:
pxrd.plot_sim(
    cof_name=COF_NAME,
    mode=MODE,
    dft=False,
    xy_folder=None,
    xlim=(1.5, 60.0),
    show=True,
    save=True,
)

In [ ]:
pxrd.plot_sim(
    cof_name=COF_NAME,
    mode=MODE,
    dft=False,
    xy_folder=f"{COF_NAME}/5_{COF_NAME}_analysis/pxrd_xy",
    output_folder=f"{COF_NAME}/5_{COF_NAME}_analysis",
    xlim=(1.5, 30.0),
    show=True,
    save=True,
)

> 🚨 For a working example of `PXRD.plot_sim_vs_exp()`, please refer to the hybrid workflow example. Experimental `.xy` data for COF-1 were not available for me , so this functionality cannot be demonstrated here. But example code is demonstrated below.

### PXRD Plot

- `plot_sim_vs_exp`  
  One experimental `.xy` file compared against every simulated `.xy` file in its own row.

#### Parameters

- `xlim` *(tuple[float, float], optional, default: `(1.5, 60.0)`)*  
  X-axis bounds as minimum and maximum 2θ range [º].


#### Output

- `{COF_NAME}_{serr|incl|both}.png`.

In [ ]:
pxrd = cl.PXRD()
pxrd.plot_sim_vs_exp(
    cof_name=COF_NAME,
    mode=MODE,
    xlim=(1.5, 30.0),
)